# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
from dotenv import load_dotenv
import os

load_dotenv(r"C:\users\Lenovo\Gitrepo\RAGAgenticAIRepo\src\.env")

print(os.getenv("OPEN_AI_API_KEY") is not None)

True


In [ ]:
import asyncio
import json
import os
from dotenv import load_dotenv
import time
from pathlib import Path
from pprint import pprint

import re

import pandas as pd
from openai import AsyncOpenAI

load_dotenv()

#print(os.getenv("OPEN_AI_API_KEY") )
#print(os.getenv("OPEN_AI_BASE_URL") )


# Make sure your OPENAI_API_KEY is set in the environment
assert os.getenv("OPEN_AI_API_KEY"), 'Set OPENAI_API_KEY first'
assert os.getenv("OPEN_AI_BASE_URL"), 'Set OPEN_AI_BASE_URL first'

client = AsyncOpenAI(api_key=os.getenv("OPEN_AI_API_KEY"), base_url=os.getenv("OPEN_AI_BASE_URL"))

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

## Step 1 — Load the data

In [3]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [27]:
from urllib import response


def prompt_zero_shot(snippet_text: str) -> list[dict]:

    system_instruction = """
    You are an expert information extraction assistant specializing in
    job descriptions.

    Your task is to accurately extract structured information from the
    provided job snippet.

    Rules:
        - Extract information only from the provided snippet.
        - Do not guess or invent information.
        - Use null when information is unavailable.
        - years_experience_required must be a number or null.
        - Return only valid JSON.
    """

    user_prompt = f"""
        Extract the following information from this job snippet.

    Job snippet:
    {snippet_text}

    Return JSON with exactly these fields:

    {{
        "id": null,
        "company": null,
         "Role": null,
        "years_experience_required": null,
        "notes": null
    }}
    """

    return [
        {
            "role": "system",
            "content": system_instruction
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
        

 


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """
    Generate a few-shot prompt for structured job-data extraction.
    The examples teach the model how to map job-snippet text
    into the required JSON structure.
    """

    system_instruction = """
    You are an expert information extraction assistant specializing in
    job descriptions.

    Your task is to accurately extract structured information from job
    snippets.

    Rules:
        - Extract information only from the provided snippet.
        - Do not guess or invent information.
        - Use null when information is unavailable.
        - years_experience_required must be a number or null.
        - Preserve the meaning of the source text.
        - Return only valid JSON.
"""

    user_prompt = f"""
Extract structured information from the job snippet.

Use the examples below as guidance for how the information should
be extracted.

========================
EXAMPLE 1
========================

Job snippet:
{{
    "id": "JD01",
    "snippet": "Amazon is hiring a Senior Software Engineer.
    The candidate must have 6+ years of software development experience."
}}

Expected output:
{{
    "id": "JD01",
    "company": "Amazon",
    "Role": "Senior Software Engineer",
    "years_experience_required": 6,
    "notes": null
}}

========================
EXAMPLE 2
========================

Job snippet:
{{
    "id": "JD02",
    "snippet": "Microsoft is looking for a Software Engineer
    with experience in Python and Azure. Knowledge of distributed
    systems is preferred."
}}

Expected output:
{{
    "id": "JD02",
    "company": "Microsoft",
    "Role": "Software Engineer",
    "years_experience_required": null,
    "notes": "Experience with Python and Azure; knowledge of distributed systems is preferred."
}}


========================
NOW PROCESS THIS SNIPPET
========================

Job snippet:
{snippet_text}

Return a JSON object with exactly these fields:

{{
    "id": null,
    "company": null,
    "Role": null,
    "years_experience_required": null,
    "notes": null
}}

Rules:
- Use the job snippet ID as "id".
- Extract the company name as "company".
- Extract the job title as "Role".
- Extract the required years of experience as a number.
- If no required experience is stated, use null.
- Do not treat preferred experience as required experience.
- Put other useful job-related information in "notes".
- Do not guess missing information.
- Return only valid JSON.
"""

    return [
        {
            "role": "system",
            "content": system_instruction
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]



def prompt_structured(snippet_text: str) -> list[dict]:
    """
    Strategy 3 — structured / role-based.

    Uses a recruiter persona, explicit field definitions,
    extraction rules, and a JSON schema.
    """

    system_instructions = """
    You are an expert technical recruiter and job-data extraction specialist.

    Your task is to analyze job descriptions and extract accurate,
    structured information.

    Follow these requirements strictly:

        1. Extract information only from the provided job snippet.
        2. Never guess, infer, or invent information that is not explicitly stated.
        3. Use null when a field cannot be determined from the snippet.
        4. Treat "5+ years" as 5 years.
        5. Extract only explicitly required experience for
        years_experience_required.
        6. Do not treat preferred or desirable experience as required experience.
        7. Return only the requested JSON object.
        8. Do not include explanations, markdown, or additional fields.

    JSON schema:

        {
            "id": "string or null",
            "company": "string or null",
            "Role": "string or null",
            "years_experience_required": "number or null",
            "notes": "string or null"
        }

        Field definitions:

            - id:
                The unique identifier of the job snippet.

            - company:
                The name of the company offering the job.

            - Role:
                The job title or position being advertised.

            - years_experience_required:
                The minimum number of years of experience explicitly required
                for the role. Return only a number. Return null if not specified.

            - notes:
                Other relevant information from the snippet, such as required
                technical skills, technologies, qualifications, or observations.
    """

    user_prompt = f"""
        Analyze the following job snippet and extract the required fields
        according to the instructions and schema provided by the system.

    Job snippet:
    {snippet_text}

    Return exactly this JSON structure:

        {{
            "id": null,
            "company": null,
            "Role": null,
            "years_experience_required": null,
            "notes": null
        }}
    """

    return [
        {
            "role": "system",
            "content": system_instructions
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
    


def prompt_cot(snippet_text: str) -> list[dict]:

    """
    Strategy 4 — reasoning-guided extraction.

    Ask the model to internally reason through the extraction task
    before returning the final structured JSON.
    """

    system_instructions = """
    You are an expert information extraction assistant specializing in
    job descriptions.

    Your task is to carefully analyze a job snippet and extract accurate
    structured information.

    Before producing the final answer, reason through the extraction
    internally.

    Follow this reasoning process:

        1. Identify the job snippet ID.
        2. Identify the company name.
        3. Identify the job title or role.
        4. Identify whether the snippet explicitly states required years
        of experience.
        5. If experience is stated as "5+ years", return 5.
        6. Do not infer years of experience from the seniority of the role.
        7. Do not treat preferred or desirable experience as required experience.
        8. Identify other relevant technical skills or qualifications.
        9. Verify that every extracted value is supported by the snippet.
        10. Use null when information is unavailable.

    Do not reveal your internal reasoning or chain of thought.

    Return only the final JSON object.

    The JSON must contain exactly these fields:

        {
            "id": "string or null",
            "company": "string or null",
            "Role": "string or null",
            "years_experience_required": "number or null",
            "notes": "string or null"
        }
    """

    user_prompt = f"""
    Analyze the following job snippet carefully.

    Use the reasoning process defined in the system instructions,
    but do not provide your reasoning or analysis in the response.

    Job snippet:
    {snippet_text}

    After completing your internal analysis, return only the final
    JSON object in exactly this format:

    {{
        "id": null,
        "company": null,
        "Role": null,
        "years_experience_required": null,
        "notes": null
    }}
    """

    return [
        {
            "role": "system",
            "content": system_instructions
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
    
STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [ ]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse."""
    if not text:
        return None

    text = text.strip()
    
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    try:
        result = json.loads(text)

        # We expect a JSON object/dict
        if isinstance(result, dict):
            return result

        return None

    except (json.JSONDecodeError, TypeError):
        return None
    


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""

    snippet_text = json.dumps(
        snippet,
        ensure_ascii=False,
        indent=2
    )

    prompt_fn = STRATEGIES[strategy_name]
    user_prompt = prompt_fn(snippet_text[0])
    # Take the start time before making the API call
    start_time = time.perf_counter()

    try:
        response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=prompt_fn(snippet_text),
            response_format={"type": "json_object"},
            temperature=0,
     )

        elapsed_seconds = time.perf_counter() - start_time

        # Get model response
        raw_text = response.choices[0].message.content or ""
        print(print("RAW MODEL RESPONSE:", raw_text))

        # Parse JSON response
        parsed = parse_response(raw_text)

        # Get token usage
        usage = response.usage

        input_tokens = usage.prompt_tokens if usage else 0
        output_tokens = usage.completion_tokens if usage else 0
        total_tokens = usage.total_tokens if usage else 0
         # Compute estimated cost
        rate = RATES[MODEL]

        cost_usd = (
            input_tokens * rate["in"]
            + output_tokens * rate["out"]
        )
        return {
            "strategy": strategy_name,
            "snippet_id": snippet.get("id"),
            "response": parsed,
            "raw_response": raw_text,
            "latency_s": elapsed_seconds,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "cost_usd": cost_usd,
            "success": parsed is not None,
            "error": None,
        }
    except Exception as e:
        elapsed_seconds = time.perf_counter() - start_time

        return {
            "strategy": strategy_name,
            "snippet_id": snippet.get("id"),
            "response": None,
            "raw_response": None,
            "latency_s": elapsed_seconds,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "cost_usd": 0.0,
            "success": False,
            "error": str(e),
        }


  


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = []

    # Create one task for every snippet × strategy combination
    for snippet in snippets:
        for strategy_name in STRATEGIES:
            tasks.append(
                run_one(strategy_name, snippet)
            )

    # Run all API calls concurrently
    results = await asyncio.gather(*tasks)

    return results

In [ ]:
# Run it
results = []

for snippet in snippets:
    result = await run_one("cot", snippet)
    results.append(result)
print(f'Got {len(results)} results.')
# Use a pretty printer to display the results in a readable format
pprint([r["response"] for r in results])

STRATEGY: cot
SNIPPET: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}
RAW MODEL RESPONSE: {
    "id": "j01",
    "company": "Acme Corp",
    "Role": "Senior Software Engineer",
    "years_experience_required": 5,
    "notes": "Strong skills in Python and distributed systems."
}
None
STRATEGY: cot
SNIPPET: {'id': 'j02', 'snippet': "We're looking for a Data Analyst at Northwind Ltd. Reporting to the Head of Analytics, you'll work with SQL, dashboards, and stakeholder requests. 2 years minimum experience preferred."}
RAW MODEL RESPONSE: {
    "id": "j02",
    "company": "Northwind Ltd.",
    "Role": "Data Analyst",
    "years_experience_required": null,
    "notes": "2 years minimum experience preferred."
}
None
STRATEGY: cot
SNIPPET: {'id': 'j03', 'snippet': "Globex International seeks a Product Manager - Growth. Yo

In [ ]:
# Run all strategies on all snippets concurrently
results = await run_all()

print(f"Total results: {len(results)}")
#User a pretty printer to display the results in a readable format
pprint([r["response"] for r in results])



STRATEGY: zero_shot
SNIPPET: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}
STRATEGY: few_shot
SNIPPET: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}
STRATEGY: structured
SNIPPET: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}
STRATEGY: cot
SNIPPET: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}
ST

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [31]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    """
    Compare 3 fields:
      - company
      - Role
      - years_experience_required

    String comparisons are case-insensitive and whitespace-trimmed.
    Return a score from 0 to 3.
    """

    if extracted is None:
        return 0
    # Handle Role/role difference
    extracted_role = extracted.get("Role", extracted.get("role"))
    gold_role = gold.get("role", gold.get("Role"))

    comparisons = [
        (extracted.get("company"), gold.get("company")),
        (extracted_role, gold_role),
        (
            extracted.get("years_experience_required"),
            gold.get("years_experience_required")
        )
    ]
   
    score = 0

    for extracted_value, gold_value in comparisons:

        if isinstance(extracted_value, str) and isinstance(gold_value, str):
            if extracted_value.strip().lower() == gold_value.strip().lower():
                score += 1

        elif extracted_value == gold_value:
            score += 1

    return score


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    
    Returns an integer from 1 to 4.

    """
    judge_system_prompt = """
    You are an expert evaluator of structured job-data extraction.

    Your task is to evaluate an extracted answer against the gold/reference
    answer and the original job snippet.

    Evaluate these three fields:
        - company
        - Role
        - years_experience_required

    Use the following rubric:

        4 — All three fields are correct.

        3 — Two of the three fields are correct AND there is no fabricated
        information.

        2 — Only one of the three fields is correct OR the extracted answer
        contains fabricated information.

        1 — None of the fields are correct OR the extracted answer is
        missing/unparsable.

    Evaluation rules:

        - Compare strings case-insensitively.
        - Ignore leading/trailing whitespace.
        - Treat equivalent numeric values as equal.
        - "5+ years" and 5 are considered equivalent.
        - Do not give credit for information that is not supported by the
    original snippet.
        - Missing information represented by null is acceptable when the
        gold answer is also null.
        - Fabricated or unsupported information should reduce the score.
        - Evaluate the answer against the gold answer, not your own assumptions.
        - Return only valid JSON.
        - Do not provide a detailed explanation or chain of thought.

    Return exactly:

    {
        "score": 1
    }
    """

    judge_user_prompt = f"""
    Evaluate the following extraction.

    ========================
    ORIGINAL JOB SNIPPET
    ========================

    {snippet_text}

    ========================
    GOLD / REFERENCE ANSWER
    ========================

    {json.dumps(gold, ensure_ascii=False, indent=2)}

    ========================
    MODEL EXTRACTED ANSWER
    ========================

    {json.dumps(extracted, ensure_ascii=False, indent=2)}

    ========================
    TASK
    ========================

    Compare the extracted answer with the gold answer using the rubric
    provided in the system instructions.

    Return only:

    {{
        "score": 1
    }}
    """

    try:
        response = await client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": judge_system_prompt
                },
                {
                    "role": "user",
                    "content": judge_user_prompt
                }
            ],
            response_format={"type": "json_object"},
            temperature=0
        )

        raw_response = response.choices[0].message.content or ""

        result = json.loads(raw_response)

        score = result.get("score")

        # Validate the judge's response
        if isinstance(score, int) and 1 <= score <= 4:
            return score

        return 1

    except (json.JSONDecodeError, TypeError, ValueError):
        return 1

    except Exception as e:
        print(f"LLM judge error: {e}")
        return 1

    

In [32]:
# Apply scoring to all 40 results
scored = []

for result in results:

    snippet_id = result["snippet_id"]

    # Find matching gold record by ID
    gold = next(
        (
            item for item in golden.values()
            if item["id"] == snippet_id
        ),
        None
    )

    if gold is None:
        print(f"Warning: No gold data found for {snippet_id}")
        continue

    # Model's extracted response
    extracted = result.get("response")

    # Exact accuracy score
    accuracy = score_accuracy(
        extracted,
        gold
    )

    # Check whether parsing was successful
    parse_success = (
        extracted is not None
        and isinstance(extracted, dict)
    )

    # Find original snippet
    snippet = next(
        (
            s for s in snippets
            if s["id"] == snippet_id
        ),
        None
    )

    if snippet is None:
        print(f"Warning: No snippet found for {snippet_id}")
        continue

    snippet_text = json.dumps(
        snippet,
        ensure_ascii=False,
        indent=2
    )

    # LLM judge score
    llm_judge_score = await score_llm_judge(
        snippet_text,
        extracted,
        gold
    )

    # Copy original result and attach scoring information
    scored_result = result.copy()

    scored_result["accuracy"] = accuracy
    scored_result["parse_success"] = parse_success
    scored_result["llm_judge_score"] = llm_judge_score

    scored.append(scored_result)

print(f"Scored {len(scored)} results.")


Scored 40 results.


## Step 5 — Build the comparison table

In [35]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary


,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.7,1.0,3.8,0.001,2.440
few_shot,2.9,1.0,3.9,0.001,2.519
structured,2.8,1.0,3.9,0.001,2.209
zero_shot,2.8,1.0,3.8,0.001,2.287


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```